In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, re, time, math
from pathlib import Path
from collections import Counter, defaultdict

PROJECT_DIR = Path("/content/drive/MyDrive/421-Project")
TRAIN_FILE = PROJECT_DIR / "musique_ans_v1.0_train.jsonl"
DEV_FILE = PROJECT_DIR / "musique_ans_v1.0_dev.jsonl"

print("Project dir exists:", PROJECT_DIR.exists())

Mounted at /content/drive
Project dir exists: True


In [ ]:
def load_musique_jsonl(file_path, max_examples=None):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_examples is not None and i >= max_examples:
                break
            ex = json.loads(line)
            data.append(ex)
    return data

def paragraph_to_text(p):
    title = p.get("title", "").strip()
    text = p.get("paragraph_text", "").strip()
    return f"Title: {title}\n{text}"

def build_all_paragraphs_context(example):
    paras = example["paragraphs"]
    return "\n\n".join(paragraph_to_text(p) for p in paras)

def build_supporting_paragraphs_context(example):
    paras = [p for p in example["paragraphs"] if p.get("is_supporting") is True]
    return "\n\n".join(paragraph_to_text(p) for p in paras)

def build_qa_prompt(question, context):
    return f"""Answer the question using the context below.

Question: {question}

Context:
{context}

Answer:"""

def get_hop(ex):
    eid = ex["id"]
    if eid.startswith("2hop"): return "2-hop"
    if eid.startswith("3hop"): return "3-hop"
    if eid.startswith("4hop"): return "4-hop"
    return "other"

In [ ]:
!pip -q install rank_bm25

from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    text = text.lower()
    return re.findall(r"\w+", text)

def build_bm25_topk_context(example, k=5):
    paragraphs = example["paragraphs"]
    para_texts = [paragraph_to_text(p) for p in paragraphs]
    tokenized_corpus = [simple_tokenize(t) for t in para_texts]
    bm25 = BM25Okapi(tokenized_corpus)
    query_tokens = simple_tokenize(example["question"])
    scores = bm25.get_scores(query_tokens)
    ranked = sorted(zip(paragraphs, para_texts, scores), key=lambda x: x[2], reverse=True)
    topk = ranked[:k]
    topk_context = "\n\n".join(x[1] for x in topk)
    return topk_context, topk

def split_paragraph_to_sentences(paragraph):
    title = paragraph.get("title", "").strip()
    text = paragraph.get("paragraph_text", "").strip()
    is_supporting = paragraph.get("is_supporting", False)
    raw_sents = re.split(r'(?<=[.!?])\s+', text)
    sentences = []
    for s in raw_sents:
        s = s.strip()
        if len(s) > 10:
            sentences.append({
                "text": s, "title": title,
                "is_supporting": is_supporting,
                "source_paragraph": paragraph
            })
    return sentences

def build_bm25_sentence_context(example, k=10):
    all_sentences = []
    for p in example["paragraphs"]:
        all_sentences.extend(split_paragraph_to_sentences(p))
    if not all_sentences:
        return "", [], []
    tokenized_sents = [simple_tokenize(s["text"]) for s in all_sentences]
    bm25 = BM25Okapi(tokenized_sents)
    query_tokens = simple_tokenize(example["question"])
    scores = bm25.get_scores(query_tokens)
    ranked = sorted(zip(all_sentences, scores), key=lambda x: x[1], reverse=True)
    topk = ranked[:k]
    topk_context = "\n\n".join(f"[{s['title']}] {s['text']}" for s, _ in topk)
    return topk_context, topk, all_sentences

In [ ]:
def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return " ".join(s.split())

def compute_em(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def compute_f1(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def compute_em_with_aliases(pred, example):
    candidates = [example["answer"]] + example.get("answer_aliases", [])
    return max(compute_em(pred, gold) for gold in candidates)

def compute_f1_with_aliases(pred, example):
    candidates = [example["answer"]] + example.get("answer_aliases", [])
    return max(compute_f1(pred, gold) for gold in candidates)

In [ ]:
!pip -q install transformers accelerate sentencepiece peft

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "google/flan-t5-large"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print(f"Using device: {device}")
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")
print(f"Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Using device: cuda
Model: google/flan-t5-large
Parameters: 783M
Memory: 3.3 GB


In [ ]:
def run_model(prompt, max_tokens=32):
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens,
            num_beams=4, early_stopping=True,
            no_repeat_ngram_size=3 if max_tokens > 32 else 0
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [ ]:
# Fixed run_model with configurable max_length

def run_model(prompt, max_tokens=32, max_input_len=512):
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=max_input_len
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens,
            num_beams=4, early_stopping=True,
            no_repeat_ngram_size=3 if max_tokens > 32 else 0
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

# Quick check: how long are B1 inputs?
sample = load_musique_jsonl(DEV_FILE, max_examples=5)
for ex in sample[:3]:
    context = build_all_paragraphs_context(ex)
    prompt = build_qa_prompt(ex["question"], context)
    tokens = tokenizer(prompt, return_tensors="pt")
    print(f"Input tokens: {tokens['input_ids'].shape[1]}")

Token indices sequence length is longer than the specified maximum sequence length for this model (3289 > 512). Running this sequence through the model will result in indexing errors


Input tokens: 3289
Input tokens: 1905
Input tokens: 2789


In [ ]:
# Check token lengths for each context type
sample = load_musique_jsonl(DEV_FILE, max_examples=50)

b1_lens = []
b2_lens = []
b3_lens = []

for ex in sample:
    # B1: all paragraphs
    p1 = build_qa_prompt(ex["question"], build_all_paragraphs_context(ex))
    b1_lens.append(len(tokenizer(p1)["input_ids"]))

    # B2: BM25 top-5
    ctx2, _ = build_bm25_topk_context(ex, k=5)
    p2 = build_qa_prompt(ex["question"], ctx2)
    b2_lens.append(len(tokenizer(p2)["input_ids"]))

    # B3: Oracle
    p3 = build_qa_prompt(ex["question"], build_supporting_paragraphs_context(ex))
    b3_lens.append(len(tokenizer(p3)["input_ids"]))

print(f"B1 (All para):  avg={sum(b1_lens)//len(b1_lens)}, max={max(b1_lens)}, fits in 512: {sum(1 for l in b1_lens if l<=512)}/{len(b1_lens)}")
print(f"B2 (BM25 top5): avg={sum(b2_lens)//len(b2_lens)}, max={max(b2_lens)}, fits in 512: {sum(1 for l in b2_lens if l<=512)}/{len(b2_lens)}")
print(f"B3 (Oracle):    avg={sum(b3_lens)//len(b3_lens)}, max={max(b3_lens)}, fits in 512: {sum(1 for l in b3_lens if l<=512)}/{len(b3_lens)}")

Token indices sequence length is longer than the specified maximum sequence length for this model (3289 > 512). Running this sequence through the model will result in indexing errors


B1 (All para):  avg=2528, max=4674, fits in 512: 0/50
B2 (BM25 top5): avg=744, max=1560, fits in 512: 7/50
B3 (Oracle):    avg=418, max=1016, fits in 512: 35/50


In [ ]:
# B1: All Paragraphs — Flan-T5-Large zero-shot

subset = load_musique_jsonl(DEV_FILE, max_examples=None)
total = len(subset)

em_scores = defaultdict(list)
f1_scores = defaultdict(list)

start = time.time()

for i, ex in enumerate(subset, start=1):
    question = ex["question"]
    hop = get_hop(ex)
    context = build_all_paragraphs_context(ex)
    prompt = build_qa_prompt(question, context)
    pred = run_model(prompt)

    em_scores[hop].append(compute_em_with_aliases(pred, ex))
    f1_scores[hop].append(compute_f1_with_aliases(pred, ex))

    if i % 200 == 0:
        elapsed = time.time() - start
        all_em = [s for h in em_scores for s in em_scores[h]]
        print(f"[{i}/{total}] EM={sum(all_em)/len(all_em):.3f} Time={elapsed:.0f}s")

print("\n" + "=" * 80)
print(f"B1: All Paragraphs — Flan-T5-Large")
row = f"{'2-hop':>10} {'3-hop':>10} {'4-hop':>10} {'All EM':>10} {'All F1':>10}"
print(row)
all_em, all_f1 = [], []
vals = ""
for h in ["2-hop", "3-hop", "4-hop"]:
    vals += f" {sum(em_scores[h])/len(em_scores[h]):>10.3f}"
    all_em.extend(em_scores[h])
    all_f1.extend(f1_scores[h])
vals += f" {sum(all_em)/len(all_em):>10.3f}"
vals += f" {sum(all_f1)/len(all_f1):>10.3f}"
print(vals)
print(f"\nComparison — XXL B1: EM=0.327, F1=0.448")

[200/2417] EM=0.035 Time=76s
[400/2417] EM=0.028 Time=150s
[600/2417] EM=0.028 Time=221s
[800/2417] EM=0.026 Time=292s
[1000/2417] EM=0.027 Time=364s
[1200/2417] EM=0.024 Time=435s
[1400/2417] EM=0.024 Time=507s
[1600/2417] EM=0.024 Time=584s
[1800/2417] EM=0.026 Time=656s
[2000/2417] EM=0.025 Time=733s
[2200/2417] EM=0.025 Time=803s
[2400/2417] EM=0.025 Time=875s

B1: All Paragraphs — Flan-T5-Large
     2-hop      3-hop      4-hop     All EM     All F1
      0.030      0.028      0.007      0.025      0.055

Comparison — XXL B1: EM=0.327, F1=0.448


In [ ]:
# B2: BM25 Top-5 — Flan-T5-Large zero-shot

subset = load_musique_jsonl(DEV_FILE, max_examples=None)
total = len(subset)

em_scores = defaultdict(list)
f1_scores = defaultdict(list)

start = time.time()

for i, ex in enumerate(subset, start=1):
    question = ex["question"]
    hop = get_hop(ex)

    bm25_ctx, _ = build_bm25_topk_context(ex, k=5)
    prompt = build_qa_prompt(question, bm25_ctx)
    pred = run_model(prompt)

    em_scores[hop].append(compute_em_with_aliases(pred, ex))
    f1_scores[hop].append(compute_f1_with_aliases(pred, ex))

    if i % 200 == 0:
        elapsed = time.time() - start
        all_em = [s for h in em_scores for s in em_scores[h]]
        print(f"[{i}/{total}] EM={sum(all_em)/len(all_em):.3f} Time={elapsed:.0f}s", flush=True)

all_em = [s for h in em_scores for s in em_scores[h]]
all_f1 = [s for h in f1_scores for s in f1_scores[h]]

print("\n" + "=" * 80)
print("B2: BM25 Top-5 — Flan-T5-Large")
print(f"{'2-hop':>10} {'3-hop':>10} {'4-hop':>10} {'All EM':>10} {'All F1':>10}")
vals = ""
for h in ["2-hop", "3-hop", "4-hop"]:
    vals += f" {sum(em_scores[h])/len(em_scores[h]):>10.3f}"
vals += f" {sum(all_em)/len(all_em):>10.3f}"
vals += f" {sum(all_f1)/len(all_f1):>10.3f}"
print(vals)
print(f"\nComparison — XXL B2: EM=0.167, F1=0.270")

[200/2417] EM=0.025 Time=69s
[400/2417] EM=0.043 Time=136s
[600/2417] EM=0.047 Time=196s
[800/2417] EM=0.041 Time=261s
[1000/2417] EM=0.043 Time=327s
[1200/2417] EM=0.045 Time=395s
[1400/2417] EM=0.045 Time=458s
[1600/2417] EM=0.045 Time=522s
[1800/2417] EM=0.046 Time=588s
[2000/2417] EM=0.046 Time=650s
[2200/2417] EM=0.046 Time=717s
[2400/2417] EM=0.048 Time=780s

B2: BM25 Top-5 — Flan-T5-Large
     2-hop      3-hop      4-hop     All EM     All F1
      0.058      0.042      0.027      0.048      0.096

Comparison — XXL B2: EM=0.167, F1=0.270


In [ ]:
# B3: Oracle — Flan-T5-Large zero-shot

subset = load_musique_jsonl(DEV_FILE, max_examples=None)
total = len(subset)

em_scores = defaultdict(list)
f1_scores = defaultdict(list)

start = time.time()

for i, ex in enumerate(subset, start=1):
    question = ex["question"]
    hop = get_hop(ex)

    context = build_supporting_paragraphs_context(ex)
    prompt = build_qa_prompt(question, context)
    pred = run_model(prompt)

    em_scores[hop].append(compute_em_with_aliases(pred, ex))
    f1_scores[hop].append(compute_f1_with_aliases(pred, ex))

    if i % 200 == 0:
        elapsed = time.time() - start
        all_em = [s for h in em_scores for s in em_scores[h]]
        print(f"[{i}/{total}] EM={sum(all_em)/len(all_em):.3f} Time={elapsed:.0f}s", flush=True)

all_em = [s for h in em_scores for s in em_scores[h]]
all_f1 = [s for h in f1_scores for s in f1_scores[h]]

print("\n" + "=" * 80)
print("B3: Oracle — Flan-T5-Large")
print(f"{'2-hop':>10} {'3-hop':>10} {'4-hop':>10} {'All EM':>10} {'All F1':>10}")
vals = ""
for h in ["2-hop", "3-hop", "4-hop"]:
    vals += f" {sum(em_scores[h])/len(em_scores[h]):>10.3f}"
vals += f" {sum(all_em)/len(all_em):>10.3f}"
vals += f" {sum(all_f1)/len(all_f1):>10.3f}"
print(vals)
print(f"\nComparison — XXL B3: EM=0.491, F1=0.649")
print(f"Comparison — Large LoRA Method 5: EM=0.431, F1=0.536")

[200/2417] EM=0.290 Time=55s
[400/2417] EM=0.347 Time=111s
[600/2417] EM=0.355 Time=166s
[800/2417] EM=0.350 Time=222s
[1000/2417] EM=0.344 Time=281s
[1200/2417] EM=0.350 Time=337s
[1400/2417] EM=0.349 Time=393s
[1600/2417] EM=0.354 Time=447s
[1800/2417] EM=0.353 Time=502s
[2000/2417] EM=0.351 Time=557s
[2200/2417] EM=0.351 Time=613s
[2400/2417] EM=0.352 Time=669s

B3: Oracle — Flan-T5-Large
     2-hop      3-hop      4-hop     All EM     All F1
      0.432      0.295      0.210      0.352      0.491

Comparison — XXL B3: EM=0.491, F1=0.649
Comparison — Large LoRA Method 5: EM=0.431, F1=0.536


In [ ]:
# CoT zero-shot — Flan-T5-Large (oracle context)

def build_cot_prompt_v3(question, context):
    return f"""Answer the question using the context below.

Reason step by step, then on a new line write only "Answer:" followed by the short final answer.

Question: {question}

Context:
{context}

Reasoning:"""

def extract_answer_v3(output):
    matches = list(re.finditer(r"(?:Final\s+)?Answer:\s*(.*)", output, re.IGNORECASE))
    if matches:
        return matches[-1].group(1).strip().rstrip(".")
    match = re.search(r"(?:so,?\s+)?the\s+(?:final\s+)?answer\s+is\s+(.*)", output, re.IGNORECASE)
    if match:
        return match.group(1).strip().rstrip(".")
    first_sent = re.split(r'[.\n]', output.strip())[0].strip()
    return first_sent if first_sent else output.strip()

subset = load_musique_jsonl(DEV_FILE, max_examples=None)
total = len(subset)

em_scores = defaultdict(list)
f1_scores = defaultdict(list)

start = time.time()

for i, ex in enumerate(subset, start=1):
    question = ex["question"]
    hop = get_hop(ex)

    context = build_supporting_paragraphs_context(ex)
    prompt = build_cot_prompt_v3(question, context)
    raw = run_model(prompt, max_tokens=256)
    pred = extract_answer_v3(raw)

    em_scores[hop].append(compute_em_with_aliases(pred, ex))
    f1_scores[hop].append(compute_f1_with_aliases(pred, ex))

    if i % 200 == 0:
        elapsed = time.time() - start
        all_em = [s for h in em_scores for s in em_scores[h]]
        print(f"[{i}/{total}] EM={sum(all_em)/len(all_em):.3f} Time={elapsed:.0f}s", flush=True)

all_em = [s for h in em_scores for s in em_scores[h]]
all_f1 = [s for h in f1_scores for s in f1_scores[h]]

print("\n" + "=" * 80)
print("CoT zero-shot — Flan-T5-Large (oracle)")
print(f"{'2-hop':>10} {'3-hop':>10} {'4-hop':>10} {'All EM':>10} {'All F1':>10}")
vals = ""
for h in ["2-hop", "3-hop", "4-hop"]:
    vals += f" {sum(em_scores[h])/len(em_scores[h]):>10.3f}"
vals += f" {sum(all_em)/len(all_em):>10.3f}"
vals += f" {sum(all_f1)/len(all_f1):>10.3f}"
print(vals)
print(f"\nComparison — XXL CoT zero-shot: EM=0.515, F1=0.653")

[200/2417] EM=0.275 Time=69s
[400/2417] EM=0.315 Time=141s
[600/2417] EM=0.327 Time=212s
[800/2417] EM=0.305 Time=282s
[1000/2417] EM=0.300 Time=355s
[1200/2417] EM=0.297 Time=427s
[1400/2417] EM=0.301 Time=497s
[1600/2417] EM=0.306 Time=567s
[1800/2417] EM=0.307 Time=639s
[2000/2417] EM=0.306 Time=711s
[2200/2417] EM=0.307 Time=779s
[2400/2417] EM=0.308 Time=851s

CoT zero-shot — Flan-T5-Large (oracle)
     2-hop      3-hop      4-hop     All EM     All F1
      0.375      0.262      0.198      0.310      0.457

Comparison — XXL CoT zero-shot: EM=0.515, F1=0.653


In [ ]:
# CoT few-shot — Flan-T5-Large (oracle context)

# Load training demos (same as XXL notebook)
train_demos = load_musique_jsonl(TRAIN_FILE, max_examples=20)
demo1 = train_demos[0]
demo2 = train_demos[1]

demo1_context = build_supporting_paragraphs_context(demo1)
demo2_context = build_supporting_paragraphs_context(demo2)

def build_fewshot_cot_prompt(question, context):
    return f"""Answer the question using the context. Reason step by step, then on a new line write only "Answer:" followed by the short final answer.

Question: {demo1["question"]}

Context:
{demo1_context}

Reasoning: The question asks what {demo1["question"].lower()} First, I need to find where Nissedal is located. The context says Nissedal is a municipality in Vestfold og Telemark county, Norway. So the country is Norway. Then I need to find what Norway is named after. The context says the name Norway comes from the Old Norse word "norðrvegr" meaning "the northern way" or simply "north."
Answer: {demo1["answer"]}

Question: {demo2["question"]}

Context:
{demo2_context}

Reasoning: The question asks about the highest point in the country where Bugabula is found. First, Bugabula is located in Kamuli District, Uganda. Then I need to find the highest point in Uganda. The context says the highest point is 1,400 metres.
Answer: {demo2["answer"]}

Question: {question}

Context:
{context}

Reasoning:"""

# Check token length with a sample
sample = load_musique_jsonl(DEV_FILE, max_examples=10)
for ex in sample[:5]:
    context = build_supporting_paragraphs_context(ex)
    prompt = build_fewshot_cot_prompt(ex["question"], context)
    tokens = tokenizer(prompt, return_tensors="pt")
    print(f"Tokens: {tokens['input_ids'].shape[1]} | Q: {ex['question'][:50]}")

Tokens: 925 | Q: Where do Greyhound buses leave from in the city wh
Tokens: 752 | Q: Which county does Lloyd Dane's birthplace belong t
Tokens: 922 | Q: Who wrote "Turn Me On" by performer of "Happy Pill
Tokens: 1039 | Q: Who did the screenwriter for Good Will Hunting pla
Tokens: 931 | Q: What is the seat of the county sharing a border wi


In [ ]:
# Load Method 5 LoRA adapter for non-oracle evaluation

from peft import PeftModel

peft_model_large = PeftModel.from_pretrained(
    model,
    "/content/drive/MyDrive/421-Project/lora_method5_v2_best"
)
peft_model_large.eval()
print("LoRA adapter loaded")
print(f"Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

LoRA adapter loaded
Memory: 3.3 GB


In [ ]:
# Method 5 vs zero-shot on BM25 Top-5 context

subset = load_musique_jsonl(DEV_FILE, max_examples=None)
total = len(subset)

em_lora = defaultdict(list)
f1_lora = defaultdict(list)
em_zs = defaultdict(list)
f1_zs = defaultdict(list)

start = time.time()

for i, ex in enumerate(subset, start=1):
    question = ex["question"]
    hop = get_hop(ex)

    bm25_ctx, _ = build_bm25_topk_context(ex, k=5)

    # LoRA Method 5 with BM25 context
    prompt_lora = (f"Answer the question and identify which paragraphs support your answer.\n\n"
                   f"Question: {question}\n\nContext:\n{bm25_ctx}\n\n"
                   f"Answer and supporting paragraphs:")

    inputs = tokenizer(prompt_lora, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = peft_model_large.generate(**inputs, max_new_tokens=64, num_beams=4,
                                         no_repeat_ngram_size=3, early_stopping=True)
    raw = tokenizer.decode(out[0], skip_special_tokens=True)
    match = re.search(r"Answer:\s*(.*?)(?:\n|Supporting:|$)", raw)
    pred_lora = match.group(1).strip() if match else raw.strip()

    em_lora[hop].append(compute_em_with_aliases(pred_lora, ex))
    f1_lora[hop].append(compute_f1_with_aliases(pred_lora, ex))

    # Zero-shot with BM25 context
    prompt_zs = build_qa_prompt(question, bm25_ctx)
    inputs_zs = tokenizer(prompt_zs, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        peft_model_large.disable_adapter_layers()
        out_zs = peft_model_large.generate(**inputs_zs, max_new_tokens=32, num_beams=4, early_stopping=True)
        peft_model_large.enable_adapter_layers()
    pred_zs = tokenizer.decode(out_zs[0], skip_special_tokens=True).strip()

    em_zs[hop].append(compute_em_with_aliases(pred_zs, ex))
    f1_zs[hop].append(compute_f1_with_aliases(pred_zs, ex))

    if i % 200 == 0:
        elapsed = time.time() - start
        all_lora = [s for h in em_lora for s in em_lora[h]]
        all_zs = [s for h in em_zs for s in em_zs[h]]
        print(f"[{i}/{total}] LoRA={sum(all_lora)/len(all_lora):.3f} "
              f"ZS={sum(all_zs)/len(all_zs):.3f} Time={elapsed:.0f}s", flush=True)

print("\n" + "=" * 90)
print(f"{'Method':<35} {'2-hop':>10} {'3-hop':>10} {'4-hop':>10} {'All EM':>10} {'All F1':>10}")
print("=" * 90)

for name, em_dict, f1_dict in [("Large ZS (BM25 top-5)", em_zs, f1_zs),
                                 ("Large LoRA M5 (BM25 top-5)", em_lora, f1_lora)]:
    row = f"{name:<35}"
    all_em, all_f1 = [], []
    for h in ["2-hop", "3-hop", "4-hop"]:
        row += f" {sum(em_dict[h])/len(em_dict[h]):>10.3f}"
        all_em.extend(em_dict[h])
        all_f1.extend(f1_dict[h])
    row += f" {sum(all_em)/len(all_em):>10.3f}"
    row += f" {sum(all_f1)/len(all_f1):>10.3f}"
    print(row)

print("=" * 90)
print(f"\nComparison — Large ZS oracle: EM=0.352 | Large LoRA oracle: EM=0.431")
print(f"Comparison — XXL ZS BM25: EM=0.167")

[200/2417] LoRA=0.130 ZS=0.025 Time=414s
[400/2417] LoRA=0.140 ZS=0.043 Time=833s
[600/2417] LoRA=0.135 ZS=0.047 Time=1239s
[800/2417] LoRA=0.133 ZS=0.041 Time=1653s
[1000/2417] LoRA=0.136 ZS=0.043 Time=2068s
[1200/2417] LoRA=0.133 ZS=0.045 Time=2491s
[1400/2417] LoRA=0.134 ZS=0.045 Time=2913s
[1600/2417] LoRA=0.133 ZS=0.045 Time=3335s
[1800/2417] LoRA=0.132 ZS=0.046 Time=3750s
[2000/2417] LoRA=0.133 ZS=0.046 Time=4157s
[2200/2417] LoRA=0.133 ZS=0.046 Time=4580s
[2400/2417] LoRA=0.133 ZS=0.048 Time=4961s

Method                                   2-hop      3-hop      4-hop     All EM     All F1
Large ZS (BM25 top-5)                    0.058      0.042      0.027      0.048      0.096
Large LoRA M5 (BM25 top-5)               0.124      0.161      0.111      0.133      0.194

Comparison — Large ZS oracle: EM=0.352 | Large LoRA oracle: EM=0.431
Comparison — XXL ZS BM25: EM=0.167


In [ ]:
# Load Flan-T5-XXL in bf16 alongside Large

import gc, torch

# Keep Large model — don't delete it
# Just load XXL additionally

print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")
print(f"Current memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

XXL_MODEL = "google/flan-t5-xxl"

tokenizer_xxl = AutoTokenizer.from_pretrained(XXL_MODEL)
model_xxl = AutoModelForSeq2SeqLM.from_pretrained(
    XXL_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print(f"\nXXL loaded: {XXL_MODEL}")
print(f"Dtype: {next(model_xxl.parameters()).dtype}")
print(f"Total memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

bf16 supported: True
Current memory: 3.3 GB


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/559 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


XXL loaded: google/flan-t5-xxl
Dtype: torch.bfloat16
Total memory: 29.9 GB


In [ ]:
# LoRA on bf16 XXL — gradient check

from peft import LoraConfig, get_peft_model, TaskType
import json

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q", "v"],
)

model_xxl.enable_input_require_grads()
peft_model_xxl = get_peft_model(model_xxl, lora_config)
peft_model_xxl.print_trainable_parameters()

# Quick gradient check
peft_model_xxl.train()

# Load one training example
with open(TRAIN_FILE, "r") as f:
    ex = json.loads(f.readline())

question = ex["question"]
answer = ex["answer"]
supporting = [p for p in ex["paragraphs"] if p.get("is_supporting")]
context = "\n\n".join(paragraph_to_text(p) for p in supporting)

input_text = (f"Answer the question and identify which paragraphs support your answer.\n\n"
              f"Question: {question}\n\nContext:\n{context}\n\n"
              f"Answer and supporting paragraphs:")
titles = [p.get("title", "").strip() for p in supporting]
target_text = f"Answer: {answer}\nSupporting: {', '.join(titles)}"

inputs = tokenizer_xxl(input_text, return_tensors="pt", truncation=True, max_length=1024).to(device)
targets = tokenizer_xxl(target_text, return_tensors="pt", truncation=True, max_length=64).to(device)
labels = targets["input_ids"]
labels[labels == tokenizer_xxl.pad_token_id] = -100

outputs = peft_model_xxl(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], labels=labels)
outputs.loss.backward()

grad_a = []
grad_b = []
for name, param in peft_model_xxl.named_parameters():
    if param.requires_grad and param.grad is not None:
        if "lora_A" in name:
            grad_a.append(param.grad.norm().item())
        elif "lora_B" in name:
            grad_b.append(param.grad.norm().item())

print(f"\nLoss: {outputs.loss.item():.4f}")
print(f"lora_A avg grad norm: {sum(grad_a)/len(grad_a):.6f} (nonzero: {sum(1 for g in grad_a if g > 0)}/{len(grad_a)})")
print(f"lora_B avg grad norm: {sum(grad_b)/len(grad_b):.6f}")
print(f"Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

peft_model_xxl.zero_grad()
torch.cuda.empty_cache()

trainable params: 18,874,368 || all params: 11,285,803,008 || trainable%: 0.1672

Loss: 10.1875
lora_A avg grad norm: 0.000000 (nonzero: 0/144)
lora_B avg grad norm: 0.002921
Memory: 30.2 GB


In [ ]:
# Fix lora_B cold-start + recheck gradients

import torch.nn as nn

# Initialize lora_B with small nonzero values
for name, param in peft_model_xxl.named_parameters():
    if param.requires_grad and "lora_B" in name:
        nn.init.normal_(param.data, mean=0.0, std=0.001)

# Recheck gradients
outputs = peft_model_xxl(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], labels=labels)
outputs.loss.backward()

grad_a = []
grad_b = []
for name, param in peft_model_xxl.named_parameters():
    if param.requires_grad and param.grad is not None:
        if "lora_A" in name:
            grad_a.append(param.grad.norm().item())
        elif "lora_B" in name:
            grad_b.append(param.grad.norm().item())

print(f"Loss: {outputs.loss.item():.4f}")
print(f"lora_A avg grad norm: {sum(grad_a)/len(grad_a):.6f} (nonzero: {sum(1 for g in grad_a if g > 0)}/{len(grad_a)})")
print(f"lora_B avg grad norm: {sum(grad_b)/len(grad_b):.6f}")

peft_model_xxl.zero_grad()
torch.cuda.empty_cache()

Loss: 10.1250
lora_A avg grad norm: 0.000433 (nonzero: 144/144)
lora_B avg grad norm: 0.003967


In [ ]:
import gc, torch

# Force clear all cached memory
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Check actual allocation vs reserved
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
print(f"Allocated: {allocated:.1f} GB")
print(f"Reserved: {reserved:.1f} GB")
print(f"Gap (fragmentation): {reserved - allocated:.1f} GB")

Allocated: 41.6 GB
Reserved: 41.7 GB
Gap (fragmentation): 0.2 GB


In [ ]:
# Free Large model to make room for XXL training

import gc, torch

del model  # This is the Large model
if 'peft_model_large' in dir():
    del peft_model_large
gc.collect()
torch.cuda.empty_cache()

print(f"Memory after freeing Large: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Memory after freeing Large: 38.3 GB
